# 02.1 图像数据与变换（Image Data and Transforms）

computer vision 后，首先要搞清楚两件事：  

1. 图像在 `PyTorch` 里长什么样
2. 图像在送进模型前通常做哪些预处理

本节重点概念

- 灰度图（grayscale image）
- 彩色图（RGB image）
- 通道（channel）
- 图像张量形状（image tensor shape）
- 批次图像（batched images）
- 变换（transforms）
- 归一化（normalization）

## 学习目标

学完后你应该能

1. 理解图像张量的标准形状
2. 区分单张图像和 batch 图像
3. 使用 `torchvision.transforms` 写基础预处理流程
4. 理解为什么常常要做缩放和归一化
5. 让自定义数据集支持 transform
6. 为后续 CNN 分类任务准备数据输入

In [ ]:
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## 1. 一张图像在 `PyTorch` 里是什么形状

`PyTorch` 中图像最常见的形状约定是：  

- one grayscale image: `(C, H, W)`，其中 `C=1`
- one RGB image: `(C, H, W)`，其中 `C=3`
- 一个 batch 的图像

这和很多其他库的 `(H, W, C)` 不一样。  


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

print("images.shape =", images.shape)
print("labels.shape =", labels.shape)
print("单张原始图像 shape / one raw image shape =", images[0].shape)

`sklearn digits` 给出的单张图片 shape 是 `(8, 8)`，也就是只有高和宽。  

因为它是灰度图，所以我们通常需要显式补上通道维  


In [ ]:
raw_img = torch.tensor(images[0], dtype=torch.float32)
img_chw = raw_img.unsqueeze(0)

print("raw_img.shape =", raw_img.shape)
print("img_chw.shape =", img_chw.shape)

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(raw_img, cmap="gray")
plt.title(f"Digit / 数字: {labels[0]}")
plt.axis("off")
plt.show()

## 2. 单张图像与 batch 图像

模型通常不吃单张图，而是吃一个 batch。  

也就是说

- 单张图像
- batch 图像

In [ ]:
batch = torch.stack([
    torch.tensor(images[0], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[1], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[2], dtype=torch.float32).unsqueeze(0),
])

print("batch.shape =", batch.shape)

这里的 `batch.shape == (3, 1, 8, 8)` 表示：  

- `3` 张图片（3 images）
- 每张图 `1` 个通道（1 channel per image）
- 高 `8`（height 8）
- 宽 `8`（width 8）

In [ ]:
# 练习 1
# 把 digits 数据集中的前 5 张图像变成一个 batch。
# Turn the first 5 images in the digits dataset into one batch.
#
# 要求
# 1. 每张图先变成 (1, 8, 8)
# 2. 最终 batch 的 shape 应该是 (5, 1, 8, 8)

# imgs =
# batch5 =
# print(batch5.shape)

In [ ]:
# 练习 1 参考答案

imgs = [torch.tensor(images[i], dtype=torch.float32).unsqueeze(0) for i in range(5)]
batch5 = torch.stack(imgs)
print(batch5.shape)

## 3. 为什么要做变换

图像在进入模型前，通常会经过一串预处理。  

常见原因

- 把像素值缩放到合理范围
- 做归一化（normalize the input）
- 做数据增强（apply data augmentation）

本节先聚焦在前两者。  


In [ ]:
print("原始像素范围 / raw pixel range:", raw_img.min().item(), raw_img.max().item())

`digits` 数据集的像素范围大致在 `0` 到 `16`。  

常见做法是先缩放到 `0~1`，再做归一化  


In [ ]:
preprocess = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

img_processed = preprocess(img_chw)

print("处理前范围 / before preprocessing:", img_chw.min().item(), img_chw.max().item())
print("处理后范围 / after preprocessing:", img_processed.min().item(), img_processed.max().item())
print("处理后均值 / processed mean:", img_processed.mean().item())

`Normalize(mean=[0.5], std=[0.5])` 对单通道灰度图的意思是：  

- 每个像素先减 `0.5`
- 再除以 `0.5`

如果输入已经在 `0~1`，那输出大致就会落在 `-1~1`。  


## 4. 自定义数据集支持 transform

这一步非常关键，因为后面的真实项目里通常都会写成：  

- 数据集类（a dataset class）
- `transform` 参数（a `transform` argument）
- 在 `__getitem__` 里应用变换

In [ ]:
class DigitsImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


dataset = DigitsImageDataset(images, labels, transform=preprocess)
img0, label0 = dataset[0]
print("img0.shape =", img0.shape)
print("label0 =", label0)
print("img0.dtype =", img0.dtype)

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)
xb, yb = next(iter(loader))

print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("xb.min() =", xb.min().item())
print("xb.max() =", xb.max().item())

In [ ]:
# 练习 2
# 实现一个 transform，只做缩放到 0~1，不做 Normalize。
# Implement a transform that only scales values to 0~1, without Normalize.
#
# 然后取 dataset[0]，打印最小值和最大值。
# Then inspect dataset[0] and print the min and max values.

# simple_transform =
# simple_dataset =
# img, label =
# print(img.min().item(), img.max().item())

In [ ]:
# 练习 2 参考答案

simple_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
])
simple_dataset = DigitsImageDataset(images, labels, transform=simple_transform)
img, label = simple_dataset[0]
print(img.min().item(), img.max().item())

## 5. 关于数据增强

common data augmentation 包括：

- 随机裁剪（random crop）
- 随机旋转（random rotation）
- 颜色抖动（color jitter）
- 水平翻转（horizontal flip）

但要注意：增强不是越多越好。  

例如对数字识别来说，水平翻转往往会改变数字语义。  


## 6. 小结

这一节最重要的是把图像数据格式彻底搞清楚。  

你现在应该能回答

1. 为什么 `PyTorch` 图像常用 `(C, H, W)` 而不是 `(H, W, C)`？
2. 单张图像和 batch 图像的 shape 有什么区别？
3. 为什么图像常常要先缩放再归一化？
4. 为什么 `transform` 常写进 `Dataset`？

下一步建议

- 进入 `CNN` 基础 notebook，理解卷积层到底在对图像做什么